In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
#Read the dataset Q1_data.csv using read_csv()
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

print(f"Dataset shape: {df_food.shape}")

In [ ]:
# Task 2: Write your code here:
#Inspect the first few rows using head()
df_food.head()

In [ ]:
# Task 3: Write your code here:
#Display dataset information using info()
df_food.info()

In [ ]:
# Task 4: Write your code here:
#Show statistical description using describe()
df_food.describe()

In [ ]:
# Task 5: Write your code here:
#Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
#Drop the 'Order_ID' column from the data
cols = ['Order_ID', 'Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df_food[cols].copy()

print(f"Before: {df_clean.shape}")
df_clean = df_clean.drop(columns=['Order_ID'])
print(f"After dropping missing Order_ID: {df_clean.shape}")

In [ ]:
# Task 2: Write your code here:
#Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")
check_missing_values(df_food)


for col in ['Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']:
    df_clean[col] = df_clean[col].fillna('unknown')

In [ ]:
# Task 3: Write your code here:
#Check and remove duplicates if any exist
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(df_food)

In [ ]:
# Task 4: Write your code here:
#Encode categorical variables if needed (Bonus if used One Hot Encoding)
def encode_categorical_columns(df):
    categorical_cols = df.select_dtypes(include=["object"]).columns
    print("Categorical Columns:", list(categorical_cols))
label_encoders = encode_categorical_columns(df_food)

categorical_cols = df_food.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df_food[col] = le.fit_transform(df_food[col])
df_food

In [ ]:
# Task 5: Write your code here:
#Apply feature scaling for all features (Use StandardScaler)
data_for_scale = pd.DataFrame(df_food)
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(data_for_scale)

In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(df_food, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
#Split the dataset into features (X) and target (y)
feature_cols = ['Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
X = df_clean[feature_cols]
y = df_clean['Distance_km']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
#Use the correct split: KFold OR StratifiedKFold
#Train a RandomForest model
#Evaluate using MAE (Mean Absolute Error) ONLY
#Print the averaged score across all folds

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

# Predict and evaluate
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")


In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: